In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="fWuJ46sQNxgW5o508yPv")
project = rf.workspace("test-ikan").project("ikanikan2")
version = project.version(5)
dataset = version.download("yolov5")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 90.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ikanikan2-5 in yolov5pytorch:: 100%|██████████| 6966/6966 [00:00<00:00, 8410.11it/s]


In [2]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt


Cloning into 'yolov5'...
remote: Enumerating objects: 17577, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 17577 (delta 36), reused 7 (delta 7), pack-reused 17514 (from 4)
Receiving objects: 100% (17577/17577), 16.69 MiB | 10.40 MiB/s, done.
Resolving deltas: 100% (12034/12034), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 77.1 MB/s eta 0:00:00


In [3]:
import os, yaml

ds_root = dataset.location  # otomatis ke folder dataset roboflow
old_yaml = os.path.join(ds_root, "data.yaml")

with open(old_yaml, "r") as f:
    d = yaml.safe_load(f)

# tulis ulang dengan absolute path
new_yaml = {
    "train": os.path.join(ds_root, "train", "images"),
    "val": os.path.join(ds_root, "valid", "images"),
    "test": os.path.join(ds_root, "test", "images"),
    "nc": d["nc"],
    "names": d["names"]
}

with open(old_yaml, "w") as f:
    yaml.safe_dump(new_yaml, f, sort_keys=False)

print("✅ data.yaml fixed:", new_yaml)

# disable wandb biar gak ganggu
os.environ["WANDB_DISABLED"] = "true"


✅ data.yaml fixed: {'train': '/content/ikanikan2-5/train/images', 'val': '/content/ikanikan2-5/valid/images', 'test': '/content/ikanikan2-5/test/images', 'nc': 1, 'names': ['ikan']}


In [4]:
!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --data {dataset.location}/data.yaml \
  --weights yolov5s.pt \
  --name ikan_yolov5s


Output streaming akan dipotong hingga 5000 baris terakhir.
      87/99      4.47G    0.02054    0.03909          0        172        640:  75% 150/201 [01:01<00:28,  1.80it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      87/99      4.47G    0.02054    0.03914          0        261        640:  75% 151/201 [01:01<00:22,  2.18it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      87/99      4.47G    0.02053    0.03911          0        181        640:  76% 152/201 [01:02<00:22,  2.17it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
      87/99      4.47G 

In [5]:
!python val.py \
  --weights runs/train/ikan_yolov5s/weights/best.pt \
  --data {dataset.location}/data.yaml \
  --img 640


val: data=/content/ikanikan2-5/data.yaml, weights=['runs/train/ikan_yolov5s/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-432-g725b922e Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning /content/ikanikan2-5/valid/labels.cache... 136 images, 0 backgrounds, 0 corrupt: 100% 136/136 [00:00<?, ?it/s]
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 5/5 [00:05<00:00,  1.05s/it]
                   all        136       1191      0.876      0.798      0.884      0.632
Speed: 0.7ms pre-process, 6.8ms inference, 6.6ms NMS per image at shape (32, 3, 640, 640)
Results save

In [6]:
!python detect.py \
  --weights runs/train/ikan_yolov5s/weights/best.pt \
  --source path/ke/video_atau_folder \
  --img 640


detect: weights=['runs/train/ikan_yolov5s/weights/best.pt'], source=path/ke/video_atau_folder, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=False, save_format=0, save_csv=False, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=exp, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-432-g725b922e Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Traceback (most recent call last):
  File "/content/yolov5/detect.py", line 438, in <module>
    main(opt)
  File "/content/yolov5/detect.py", line 433, in main
    run(**vars(opt))
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 120, in decorate_context
    return

In [8]:
import shutil

# Path best.pt setelah training
src = "runs/train/v5/weights/best.pt"

# Copy/rename jadi v5.pt di root folder Colab (/content)
dst = "/content/v5.pt"
shutil.copy(src, dst)

print(f"✅ Model berhasil disalin ke {dst}")


FileNotFoundError: [Errno 2] No such file or directory: 'runs/train/v5/weights/best.pt'